# SVM Classification of chemical element data

#### <p style="text-align: right;"> &#9989; **put your name here** </p>



---
## Support-vector-machine classifier for elemental data

Here, we will load in a dataset of elements, which contains metals and nonmetals.

Each data point contains five values. The first four are atomic_radius, electron_affinity, ionization_energy, and electronegativity, respectively. The last column are labels, where the value of 1 and 0 indicates metal and nonmetal elements, respectively. The column of ionization_energy is corrupted. Thus, we will drop it. 

### Part 1. Load data and examine data.

This dataset is  obtained from [https://mds-book.org/Content/datasets](https://mds-book.org/Content/datasets), which is the supplementary data of the textbook *Materials Data Science: Introduction to Data Mining, Machine Learning, and Data-Driven Predictions for Materials Science and Engineering* (2024) by Stefan Sandfeld. The author has made a `mdsdata` Python library and a `load_elements()` method. 

1. We should intsall the `msdata` library.  The recommended way: get the package directly from PyPI by running `pip install MDSdata` on terminal environment.
2. Load the data using `load_elements()` method.
3. Convert the data to Pandas `DataFrame`, which is very convenient for examining the data.
4. Make plot to examine the clustering of elements.

In [ ]:
import matplotlib.pyplot as plt
from mdsdata import load_elements

atomic_radius, electron_affinity, \
    ionization_energy, electronegativity, label = load_elements()


Here, we grab the frist three and the last columns, and store them in a Pandas `DataFrame`.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "atomic_radius": atomic_radius,
    "electron_affinity": electron_affinity,
    "electronegativity": electronegativity,
    "metal": label})

df


Let's examine this dataset using `describe()` mehotd. 

In [ ]:
df.describe()

&#9989; Do This - What do 25%, 50%, and 75% mean in the `df.decribe()` output?

---
Let's use some Pandas tricks to find out how many entries are metal.

In [ ]:
N_metal = df[df['metal']==1].shape[0]

print('number of metallic elements =',  N_metal)

&#9989; Do This - Explain what `df[df['metal']==1]` and `.shape[0]` are doing?

---
### Visualize the distribution of data points.

First, we plot the data points with the two features: atomic_radius and electron_affinity. Use 

In [ ]:
# Names for classes
names = ["nonmetal", "metal"]

# Colors for two classes
colors = ['green', 'red']

fig, ax = plt.subplots(figsize=(4, 3))

ax.scatter(atomic_radius[label==0], electron_affinity[label==0], c='r', label=names[0])
ax.scatter(atomic_radius[label==1], electron_affinity[label==1], c='b', label=names[1])

ax.set(xlabel='atomic radius [pm]', ylabel='electron affinity [kJ/mol]')
plt.legend()
plt.show()


&#9989; Do This - By looking at this scatter plot, do you think a soft or a hard margin will do a better job? Why?

### 3D scatter plot

The code cell make a 3D scatter plot of the data. You can rotate the figure to examine the distribution of data entries (points).

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib ipympl

# Names for classes
names = ["nonmetal", "metal"]

# Colors for two classes
colors = ['green', 'red']


# y contains 0 (nonmetal) or 1 (metal)
y = df['metal'].to_numpy()

# Convert first 3 columns to numpy
X = df.iloc[:, :3].to_numpy()
x1 = X[:, 0]
x2 = X[:, 1]
x3 = X[:, 2]


fig = plt.figure(figsize=(4, 3))
ax = fig.add_subplot(111, projection='3d')

# Plot each class: 0 and 1
for cls in range(2):
    ax.scatter(
        x1[y == cls],
        x2[y == cls],
        x3[y == cls],
        label=names[cls],
        color=colors[cls],
        s=50)

ax.set_xlabel('atomic_radius')
ax.set_ylabel('electron_affinity')
ax.set_zlabel('electronegativity')
ax.set_title('Metal vs Nonmetal 3D Plot')
ax.legend()

plt.show()

&#9989; Do This - Rotate the figure and examine from different angles. If you can only use two features, which two can make the best separation of the classes? Why?

---
### Part 2. SVM classifier

Here, we will build a SVM classifier using `sklearn` library, which is widely used as a learning tool in data science courses.



In [ ]:
# load relevant libraries

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

The code below:

1. Perform `train_test_split()`: we will split the entire dataset to training data and test data. Training data are the entries used to determine hyperplane and support vectors. Test data are the entries used to evaluate the trained SVM classifier.
2. Scale the data features to have a mean at zero.
3. Call a `SVC` class (object) from  `sklearn` library.
4. Train (find) the hyperplane (dividing boundary) iterative using training data. We will use `fit()` method.
5. Predict (classify) the test data using `predict()`method.
6. Evaluate the accuracy of the train SVM.

In [ ]:
# 1. Train–test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)

# 2. (Optional but common) Feature scaling
# scaled to have a mean of 0
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 3. Define SVM classifier
# You can try different kernels: "linear", "rbf", "poly", etc.
clf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)

# 4. Train the model
clf.fit(X_train_scaled, y_train)

# 5. Make predictions
y_pred = clf.predict(X_test_scaled)

# 6. Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=names))

&#9989; Do This - The `scaler()` method scales the values of the same feature to have a mean of 0 and normalizes the values with the standard deviation. Explain in your own words. Why should the data be scaled? Do you think this step will increase the accuracy? Why?

&#9989; Do This - Do you think using more training data (higher  train_test_split ratio) will improve the accuracy? Why?

---
### Part 3 Hyperparameter tunning

In the classifier above, we can see there are three parameters tunable: 
1. kernel: The method to calculate distance in hyper dimension. Here, we select two types of it. `rbf` and `linear`.
2. margin: The soft versus hard margin.
3. gamma: smoothness of the decision boundary. A low and a high gamma leads to a smooth and a wiggly boundary, respectively. 

How do we find the best parameter set for an accurate classifier? Let's use grid search (trial-and-error test) to find the best set of parameters. Let's set a few grid points in the range of [0.01, 100] for `C`. A few grid points in the range of [0.001, 0.03] for `gamma`, and two types of kernel: ['linear','rbf']. Note that there are too many possibilities of `poly` kernels. Thus, we do not want to test those polynomial kernels. However sometimes the best kernel could be polynomials.


&#9989; Do This - What are the effects of C, gamma, and kernel? Do a internet search and answer this question.

In [ ]:
from sklearn.model_selection import GridSearchCV

# fill ?? to complete the code

#make some temporary variables so you can change this easily
tmp_vectors = X_train_scaled
tmp_labels = y_train

print("Fitting the classifier to the training set")
# a dictionary of hyperparameters: key is the name of the parameter, value is a list of values to test
param_grid = {'C': [??],
              'gamma': [??],
              'kernel': [??,??]}

# make a classifier by searching over a classifier and the parameter grid
clf = GridSearchCV(SVC(class_weight='balanced'), param_grid)

# we have a "good" classifier (according to GridSearchCV), how's it look
clf = clf.fit(tmp_vectors, tmp_labels)
print("Best estimator found by grid search:")
print(clf.best_estimator_)
print("Best parameters found by grid search:")
print(clf.best_params_)

**Evaluate the classifier with optimal parameter set.**


In [ ]:
true_labels = y_test

print("Predicting names on the test set")
pred_labels = clf.predict(X_test)

cm = confusion_matrix(true_labels, true_labels)
print("Confusion Matrix:\n", cm)

print(classification_report(true_labels, true_labels))

&#9989; Do This - What is the accuracy of the model with the best parameters?

---
### Part 4 Visualization

Use a heatmap to show the confusion matrix. 

**Plot the confusion matrix.**

In [ ]:
import seaborn as sns
target_names = names
# -------------------------
# 8. Heatmap Visualization
# -------------------------
plt.figure(figsize=(6, 5))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=target_names,
            yticklabels=target_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix Heatmap (element Classification)")
plt.show()

---
### Part 5 Prediction

Now you have a sample that has electronegativity of 3.5, electron_affinity of 165, and atomic_radius of 2.6. Use your classify to detemine whether this sample is metal.

In [ ]:
import numpy as np

X_sample = np.zeros((1,3))

X_sample[0,0] = ??
X_sample[0,1] = ??
X_sample[0,2] = ??

y_sample = ??
print(y_sample)

&#9989; Do This - what is the type of element of this sample?

## 12. Please upload your completed notebook via this [Link](https://www.dropbox.com/request/plwts1t1w6bq20c2uv0t)